In [31]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("bro it is working")


from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableBranch

bro it is working


Movie Review

In [32]:
template_review = ChatPromptTemplate.from_messages([
    ("system", " you are a movie reviewer"),
    ("human","Write a short, spoiler-free review of {movie} in one sentenece")
])

llm_model = init_chat_model(model="gpt-5-mini")

str_parser = StrOutputParser()

chain_review = template_review | llm_model | str_parser

#chain_review.invoke("KGF Chapter 1")

In [33]:
def format_review(output:str):
    return {"review" : output}

format_review_runnable = RunnableLambda(format_review)

Movie sentiment

In [34]:
template_sentiment = ChatPromptTemplate.from_messages([
    ("system", "Classify the review as Positive or Negative."),
    ("human","{review}")
])


from typing import Literal
from pydantic import BaseModel

class llm_schema(BaseModel):
    movie_summary_flag : Literal["Positive", "Negative"]

llm_structured_output = llm_model.with_structured_output(llm_schema)

chain_sentiment = template_sentiment | llm_structured_output

chain_sentiment_runnable = RunnablePassthrough.assign(sentiment=chain_sentiment)

In [35]:
chain_final_sentiment = chain_review | format_review_runnable | chain_sentiment_runnable
#chain_final_sentiment.invoke("kGF chapter 1 ")

In [36]:
template_linkedIn = ChatPromptTemplate.from_messages([
    ("system", "You create LinkedIn posts."),
    ("human", "Create one line LinkedIn post based on this movie review. {review}")])

llm_model = init_chat_model(model="gpt-5-mini")

str_parser = StrOutputParser()

chain_linkedin = template_linkedIn | llm_model | str_parser
    

In [37]:
template_Instagram = ChatPromptTemplate.from_messages([
    ("system", "You create Instagram posts."),
    ("human","Create one line Instagram post based on this movie review. {review}")])

llm_model = init_chat_model(model="gpt-5-mini")

str_parser = StrOutputParser()

chain_instagram = template_Instagram | llm_model | str_parser

In [39]:
def is_positive(data: dict):
    sentiment = data["sentiment"].movie_summary_flag

    return sentiment == "Positive"


conditional_chain = RunnableBranch(
    (is_positive, chain_linkedin),
    chain_instagram,
)

conditional_chain_runnable = RunnablePassthrough.assign(post = conditional_chain)

In [40]:
movie_chain = (
    chain_final_sentiment | conditional_chain_runnable    
)

result = movie_chain.invoke({"movie": "KGF Chpater 1"})

print(result)

{'review': 'KGF Chapter 1 is a stylish, high-octane action drama anchored by Yash’s magnetic presence, gritty visuals, and a thunderous score that make for a visceral, if occasionally heavy-handed, cinematic ride.', 'sentiment': llm_schema(movie_summary_flag='Positive'), 'post': 'KGF Chapter 1 delivers a stylish, high-octane cinematic ride—Yash’s magnetic presence, gritty visuals, and a thunderous score make it visceral, even if occasionally heavy-handed.'}
